# Predicting Smartphone Addiction — Modeling

This notebook uses leakage-safe stratified cross-validation and optimizes ROC AUC. It trains LightGBM and CatBoost by default, produces OOF predictions for blending, and writes a Kaggle submission.

In [1]:
import os
import subprocess
from pathlib import Path

import kagglehub
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

RANDOM_SEED = 42
N_SPLITS = 5
# Add 'xgboost' only after the baseline; it is substantially slower.
MODELS_TO_RUN = ('lightgbm', 'catboost', 'xgboost')

def cuda_gpu_available() -> bool:
    """Return whether a CUDA GPU is visible to this notebook runtime."""
    if os.environ.get('CUDA_VISIBLE_DEVICES') in {'', '-1'}:
        return False
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            check=True, capture_output=True, text=True, timeout=5,
        )
        return bool(result.stdout.strip())
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return False

USE_GPU = cuda_gpu_available()
LGBM_DEVICE_PARAMS = {'device_type': 'gpu'} if USE_GPU else {}
CATBOOST_DEVICE_PARAMS = {'task_type': 'GPU', 'devices': '0'} if USE_GPU else {}
XGBOOST_DEVICE_PARAMS = {'device': 'cuda'} if USE_GPU else {}
CATBOOST_EVAL_METRIC = 'AUC' if USE_GPU else 'AUC'
print(f'CUDA GPU available: {USE_GPU}; training on {"GPU" if USE_GPU else "CPU"}.')

DATA_DIR = Path(kagglehub.competition_download('playground-series-s6e8'))
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'addicted_label'
ID_COLUMN = 'id'
assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
assert sample_submission[ID_COLUMN].equals(test[ID_COLUMN])
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target rate: {train[TARGET].mean():.4f}')

CUDA GPU available: False; training on CPU.
Train: (691369, 14) | Test: (296302, 13)
Target rate: 0.7094


In [7]:
NUMERIC_COLUMNS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time',
]
CATEGORICAL_COLUMNS = ['gender', 'stress_level', 'academic_work_impact']

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create target-free features; safe to apply before CV."""
    X = frame.drop(columns=[TARGET, ID_COLUMN], errors='ignore').copy()
    raw_columns = X.columns.tolist()

    # Preserve missingness explicitly; native tree models also receive numeric NaNs.
    for column in raw_columns:
        X[f'{column}_is_missing'] = X[column].isna().astype('int8')
    X['missing_count'] = X[raw_columns].isna().sum(axis=1).astype('int8')

    # CatBoost needs categorical missing values represented as strings.
    for column in CATEGORICAL_COLUMNS:
        X[column] = X[column].astype('string').fillna('Missing')

    social, gaming = X['social_media_hours'], X['gaming_hours']
    daily, weekend = X['daily_screen_time_hours'], X['weekend_screen_time']
    work_study, sleep = X['work_study_hours'], X['sleep_hours']
    X['social_gaming_hours'] = social + gaming
    X['weekday_weekend_screen_difference'] = weekend - daily
    X['weekend_daily_screen_ratio'] = weekend / (daily + 0.25)
    X['usage_hours_total'] = daily + social + gaming + work_study
    X['screen_to_sleep_ratio'] = daily / (sleep + 0.25)

    for column in ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                   'work_study_hours', 'weekend_screen_time']:
        X[f'{column}_squared'] = X[column] ** 2
    return X

X_train = build_features(train)
X_test = build_features(test)
y = train[TARGET].astype('int8')
assert X_train.columns.tolist() == X_test.columns.tolist()
print(f'Model features: {X_train.shape[1]}')

Model features: 35


In [11]:
def make_lightgbm_input(train_features: pd.DataFrame, test_features: pd.DataFrame):
    """Set identical category levels for LightGBM."""
    train_input, test_input = train_features.copy(), test_features.copy()
    for column in CATEGORICAL_COLUMNS:
        categories = pd.Index(pd.concat([train_input[column], test_input[column]], ignore_index=True).unique())
        train_input[column] = pd.Categorical(train_input[column], categories=categories)
        test_input[column] = pd.Categorical(test_input[column], categories=categories)
    return train_input, test_input

def make_catboost_input(features: pd.DataFrame) -> pd.DataFrame:
    data = features.copy()
    for column in CATEGORICAL_COLUMNS:
        data[column] = data[column].astype(str)
    return data

X_lgb, X_test_lgb = make_lightgbm_input(X_train, X_test)
X_cat, X_test_cat = make_catboost_input(X_train), make_catboost_input(X_test)
# One-hot encoding is used only for the optional XGBoost model.
xgb_combined = pd.get_dummies(pd.concat([X_train, X_test], axis=0), columns=CATEGORICAL_COLUMNS, dtype=np.int8)
X_xgb, X_test_xgb = xgb_combined.iloc[:len(X_train)], xgb_combined.iloc[len(X_train):]
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED).split(X_train, y))

In [ ]:
def run_lightgbm_cv(X, X_submission, y, folds):
    oof, test_prediction = np.zeros(len(X)), np.zeros(len(X_submission))
    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        model = lgb.LGBMClassifier(
            objective='binary', n_estimators=5000, learning_rate=0.03, num_leaves=63,
            min_child_samples=80, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0,
            random_state=RANDOM_SEED + fold, n_jobs=-1, verbosity=-1, **LGBM_DEVICE_PARAMS,
        )
        model.fit(
            X.iloc[train_index], y.iloc[train_index],
            eval_set=[(X.iloc[valid_index], y.iloc[valid_index])], eval_metric='auc',
            categorical_feature=CATEGORICAL_COLUMNS,
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)],
        )
        oof[valid_index] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_prediction += model.predict_proba(X_submission)[:, 1] / len(folds)
        print(f'LightGBM fold {fold}: {roc_auc_score(y.iloc[valid_index], oof[valid_index]):.6f} | trees: {model.best_iteration_}')
    print(f'LightGBM OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

def run_catboost_cv(X, X_submission, y, folds):
    oof, test_prediction = np.zeros(len(X)), np.zeros(len(X_submission))
    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        model = CatBoostClassifier(
            loss_function='Logloss', eval_metric=CATBOOST_EVAL_METRIC, iterations=5000, learning_rate=0.085,
            depth=8, l2_leaf_reg=5.0, random_seed=RANDOM_SEED + fold,
            verbose=250, allow_writing_files=False, **CATBOOST_DEVICE_PARAMS,
        )
        model.fit(
            X.iloc[train_index], y.iloc[train_index], cat_features=CATEGORICAL_COLUMNS,
            eval_set=(X.iloc[valid_index], y.iloc[valid_index]), early_stopping_rounds=200,
        )
        oof[valid_index] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_prediction += model.predict_proba(X_submission)[:, 1] / len(folds)
        print(f'CatBoost fold {fold}: {roc_auc_score(y.iloc[valid_index], oof[valid_index]):.6f} | trees: {model.get_best_iteration()}')
    print(f'CatBoost OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

def run_xgboost_cv(X, X_submission, y, folds):
    oof, test_prediction = np.zeros(len(X)), np.zeros(len(X_submission))
    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        model = XGBClassifier(
            objective='binary:logistic', eval_metric='auc', n_estimators=5000,
            learning_rate=0.03, max_depth=8, min_child_weight=8,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=3.0,
            tree_method='hist', early_stopping_rounds=200,
            random_state=RANDOM_SEED + fold, n_jobs=-1, **XGBOOST_DEVICE_PARAMS,
        )
        model.fit(X.iloc[train_index], y.iloc[train_index],
                  eval_set=[(X.iloc[valid_index], y.iloc[valid_index])], verbose=250)
        oof[valid_index] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_prediction += model.predict_proba(X_submission)[:, 1] / len(folds)
        print(f'XGBoost fold {fold}: {roc_auc_score(y.iloc[valid_index], oof[valid_index]):.6f} | trees: {model.best_iteration}')
    print(f'XGBoost OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

In [ ]:
import warnings
warnings.filterwarnings('ignore')

oof_predictions, test_predictions = {}, {}

if 'lightgbm' in MODELS_TO_RUN:
    oof_predictions['lightgbm'], test_predictions['lightgbm'] = run_lightgbm_cv(X_lgb, X_test_lgb, y, folds)
if 'catboost' in MODELS_TO_RUN:
    oof_predictions['catboost'], test_predictions['catboost'] = run_catboost_cv(X_cat, X_test_cat, y, folds)
if 'xgboost' in MODELS_TO_RUN:
    oof_predictions['xgboost'], test_predictions['xgboost'] = run_xgboost_cv(X_xgb, X_test_xgb, y, folds)

scores = pd.Series({name: roc_auc_score(y, prediction) for name, prediction in oof_predictions.items()})
display(scores.sort_values(ascending=False).to_frame('OOF ROC AUC'))
pd.DataFrame({TARGET: y, **oof_predictions}).to_csv(OUTPUT_DIR / 'oof_predictions.csv', index=False)
pd.DataFrame({**X_test, **test_predictions}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[250]	valid_0's auc: 0.955406	valid_0's binary_logloss: 0.244981
[500]	valid_0's auc: 0.9599	valid_0's binary_logloss: 0.231942
[750]	valid_0's auc: 0.961068	valid_0's binary_logloss: 0.228404
[1000]	valid_0's auc: 0.961522	valid_0's binary_logloss: 0.226972
[1250]	valid_0's auc: 0.961984	valid_0's binary_logloss: 0.225531
[1500]	valid_0's auc: 0.962258	valid_0's binary_logloss: 0.224659
[1750]	valid_0's auc: 0.962466	valid_0's binary_logloss: 0.223972
[2000]	valid_0's auc: 0.962609	valid_0's binary_logloss: 0.223465
[2250]	valid_0's auc: 0.962725	valid_0's binary_logloss: 0.223063
[2500]	valid_0's auc: 0.962859	valid_0's binary_logloss: 0.222631
[2750]	valid_0's auc: 0.962975	valid_0's binary_logloss: 0.222253
[3000]	valid_0's auc: 0.963006	valid_0's binary_logloss: 0.222124
[3250]	valid_0's auc: 0.962997	valid_0's binary_logloss: 0.222108
LightGBM fold 1: 0.963017 | trees: 3159


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[250]	valid_0's auc: 0.956553	valid_0's binary_logloss: 0.242895
[500]	valid_0's auc: 0.96077	valid_0's binary_logloss: 0.230159
[750]	valid_0's auc: 0.961762	valid_0's binary_logloss: 0.227008
[1000]	valid_0's auc: 0.962233	valid_0's binary_logloss: 0.22548
[1250]	valid_0's auc: 0.962652	valid_0's binary_logloss: 0.224134
[1500]	valid_0's auc: 0.962899	valid_0's binary_logloss: 0.223271
[1750]	valid_0's auc: 0.963093	valid_0's binary_logloss: 0.222611
[2000]	valid_0's auc: 0.96321	valid_0's binary_logloss: 0.222206
[2250]	valid_0's auc: 0.963341	valid_0's binary_logloss: 0.221759
[2500]	valid_0's auc: 0.96346	valid_0's binary_logloss: 0.221359
[2750]	valid_0's auc: 0.963555	valid_0's binary_logloss: 0.221027
[3000]	valid_0's auc: 0.963625	valid_0's binary_logloss: 0.220796
[3250]	valid_0's auc: 0.963656	valid_0's binary_logloss: 0.220666
[3500]	valid_0's auc: 0.963661	valid_0's binary_logloss: 0.220633
LightGBM fold 2: 0.963675 | trees: 3372


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

[250]	valid_0's auc: 0.957971	valid_0's binary_logloss: 0.239869
[500]	valid_0's auc: 0.962144	valid_0's binary_logloss: 0.226845
[750]	valid_0's auc: 0.963106	valid_0's binary_logloss: 0.223651
[1000]	valid_0's auc: 0.963619	valid_0's binary_logloss: 0.221928
[1250]	valid_0's auc: 0.964036	valid_0's binary_logloss: 0.220519
[1500]	valid_0's auc: 0.964253	valid_0's binary_logloss: 0.219732
[1750]	valid_0's auc: 0.964471	valid_0's binary_logloss: 0.218944
[2000]	valid_0's auc: 0.964603	valid_0's binary_logloss: 0.218422
[2250]	valid_0's auc: 0.964642	valid_0's binary_logloss: 0.218199
[2500]	valid_0's auc: 0.96473	valid_0's binary_logloss: 0.217867
[2750]	valid_0's auc: 0.964746	valid_0's binary_logloss: 0.217759
LightGBM fold 13: 0.964754 | trees: 2674


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[250]	valid_0's auc: 0.956083	valid_0's binary_logloss: 0.243574
[500]	valid_0's auc: 0.960532	valid_0's binary_logloss: 0.230313
[750]	valid_0's auc: 0.961596	valid_0's binary_logloss: 0.226908
[1000]	valid_0's auc: 0.962078	valid_0's binary_logloss: 0.225309
[1250]	valid_0's auc: 0.962321	valid_0's binary_logloss: 0.224458
[1500]	valid_0's auc: 0.962541	valid_0's binary_logloss: 0.223706
[1750]	valid_0's auc: 0.962734	valid_0's binary_logloss: 0.223029
[2000]	valid_0's auc: 0.962879	valid_0's binary_logloss: 0.222512
[2250]	valid_0's auc: 0.962965	valid_0's binary_logloss: 0.222185
[2500]	valid_0's auc: 0.963065	valid_0's binary_logloss: 0.221835
[2750]	valid_0's auc: 0.963089	valid_0's binary_logloss: 0.221698
[3000]	valid_0's auc: 0.963103	valid_0's binary_logloss: 0.22162
[3250]	valid_0's auc: 0.963152	valid_0's binary_logloss: 0.221447
LightGBM fold 14: 0.963153 | trees: 3279


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[250]	valid_0's auc: 0.957028	valid_0's binary_logloss: 0.241371
[500]	valid_0's auc: 0.961191	valid_0's binary_logloss: 0.228748
[750]	valid_0's auc: 0.962266	valid_0's binary_logloss: 0.22539
[1000]	valid_0's auc: 0.962694	valid_0's binary_logloss: 0.223942
[1250]	valid_0's auc: 0.963032	valid_0's binary_logloss: 0.222861
[1500]	valid_0's auc: 0.963349	valid_0's binary_logloss: 0.221807
[1750]	valid_0's auc: 0.96358	valid_0's binary_logloss: 0.221041
[2000]	valid_0's auc: 0.963708	valid_0's binary_logloss: 0.22059
[2250]	valid_0's auc: 0.963757	valid_0's binary_logloss: 0.220371
[2500]	valid_0's auc: 0.963846	valid_0's binary_logloss: 0.22006
[2750]	valid_0's auc: 0.963945	valid_0's binary_logloss: 0.219723
[3000]	valid_0's auc: 0.963955	valid_0's binary_logloss: 0.219636
[3250]	valid_0's auc: 0.963967	valid_0's binary_logloss: 0.219579
[3500]	valid_0's auc: 0.963962	valid_0's binary_logloss: 0.219558
LightGBM fold 15: 0.963978 | trees: 3332
LightGBM OOF AUC: 0.964076
0:	test: 0.9108

In [ ]:
def normalized_rank(values):
    return pd.Series(values).rank(method='average').to_numpy() / (len(values) + 1)

def select_blend(oof_predictions, y, step=0.05):
    """Find blend weights from OOF predictions only; supports up to three models."""
    names = list(oof_predictions)
    if len(names) == 1:
        return names, np.array([1.0]), False, roc_auc_score(y, oof_predictions[names[0]])
    if len(names) > 3:
        raise ValueError('Blend at most three models at a time.')
    candidates, grid = [], np.arange(0, 1 + step / 2, step)
    for use_rank in (False, True):
        arrays = [normalized_rank(oof_predictions[name]) if use_rank else oof_predictions[name] for name in names]
        weights = ([(w, 1 - w) for w in grid] if len(names) == 2 else
                   [(a, b, 1 - a - b) for a in grid for b in grid if a + b <= 1])
        for weight_set in weights:
            prediction = sum(weight * values for weight, values in zip(weight_set, arrays))
            candidates.append((roc_auc_score(y, prediction), np.array(weight_set), use_rank))
    score, weights, use_rank = max(candidates, key=lambda item: item[0])
    return names, weights, use_rank, score

model_names, blend_weights, use_rank_blend, blend_auc = select_blend(oof_predictions, y)
print(f'Best OOF blend AUC: {blend_auc:.6f}')
print(f'Models: {model_names} | Weights: {blend_weights} | Rank blend: {use_rank_blend}')

Best OOF blend AUC: 0.964625
Models: ['lightgbm', 'catboost', 'xgboost'] | Weights: [0.2 0.3 0.5] | Rank blend: False


In [ ]:
blend_components = [normalized_rank(test_predictions[name]) if use_rank_blend else test_predictions[name] for name in model_names]
final_prediction = sum(weight * prediction for weight, prediction in zip(blend_weights, blend_components))

submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], TARGET: final_prediction})
assert submission.shape == sample_submission.shape
assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
assert submission[TARGET].between(0, 1).all()

submission_path = OUTPUT_DIR / 'submission_blend.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved submission to: {submission_path}')
submission.head()

Saved submission to: outputs/submission_blend.csv


,id,addicted_label
0,691369,0.999415
1,691370,0.926961
2,691371,0.931671
3,691372,0.991621
4,691373,0.998215


## Next experiments

Run this baseline once before tuning. Keep these folds fixed and compare every change using OOF ROC AUC. Tune models only after the baseline is recorded; do not tune against the public leaderboard. Add XGBoost only if it produces genuinely different OOF predictions that improve the blend.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Run only after recording the baseline above. These use three fixed folds to control
# runtime; verify each selected configuration with the full 15-fold pipeline before submission.
RUN_OPTUNA = False
OPTUNA_MODELS = ('lightgbm', 'catboost', 'xgboost')  # Remove models here to shorten a run.
N_TRIALS_PER_MODEL = 30
TUNING_FOLDS = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED).split(X_train, y))

if RUN_OPTUNA:
    import optuna

    def mean_fold_auc(model_factory, X):
        fold_scores = []
        for train_index, valid_index in TUNING_FOLDS:
            model = model_factory()
            X_train_fold, X_valid_fold = X.iloc[train_index], X.iloc[valid_index]
            y_train_fold, y_valid_fold = y.iloc[train_index], y.iloc[valid_index]
            if isinstance(model, lgb.LGBMClassifier):
                model.fit(X_train_fold, y_train_fold,
                          eval_set=[(X_valid_fold, y_valid_fold)], eval_metric='auc',
                          categorical_feature=CATEGORICAL_COLUMNS,
                          callbacks=[lgb.early_stopping(150, verbose=False)])
            elif isinstance(model, CatBoostClassifier):
                model.fit(X_train_fold, y_train_fold, cat_features=CATEGORICAL_COLUMNS,
                          eval_set=(X_valid_fold, y_valid_fold), early_stopping_rounds=150,
                          verbose=False)
            else:
                model.fit(X_train_fold, y_train_fold,
                          eval_set=[(X_valid_fold, y_valid_fold)], verbose=False)
            prediction = model.predict_proba(X_valid_fold)[:, 1]
            fold_scores.append(roc_auc_score(y_valid_fold, prediction))
        return float(np.mean(fold_scores))

    def lightgbm_objective(trial):
        parameters = {
            'objective': 'binary', 'n_estimators': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 180),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'random_state': RANDOM_SEED, 'n_jobs': -1, 'verbosity': -1, **LGBM_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: lgb.LGBMClassifier(**parameters), X_lgb)

    def catboost_objective(trial):
        parameters = {
            'loss_function': 'Logloss', 'eval_metric': CATBOOST_EVAL_METRIC, 'iterations': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
            'depth': trial.suggest_int('depth', 5, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 20.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.01, 2.0, log=True),
            'bootstrap_type': 'Bernoulli',
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'random_seed': RANDOM_SEED, 'allow_writing_files': False, 'verbose': False, **CATBOOST_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: CatBoostClassifier(**parameters), X_cat)

    def xgboost_objective(trial):
        parameters = {
            'objective': 'binary:logistic', 'eval_metric': 'auc', 'n_estimators': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 3.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'tree_method': 'hist', 'early_stopping_rounds': 150,
            'random_state': RANDOM_SEED, 'n_jobs': -1, **XGBOOST_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: XGBClassifier(**parameters), X_xgb)

    objectives = {
        'lightgbm': lightgbm_objective,
        'catboost': catboost_objective,
        'xgboost': xgboost_objective,
    }
    for model_name in OPTUNA_MODELS:
        if model_name not in objectives:
            raise ValueError(f'Unknown Optuna model: {model_name}')
        study = optuna.create_study(direction='maximize', study_name=f's6e8_{model_name}')
        study.optimize(objectives[model_name], n_trials=N_TRIALS_PER_MODEL, show_progress_bar=True)
        print(f'{model_name} best three-fold AUC: {study.best_value:.6f}')
        print(study.best_params)


## Experiments 2
Blending with diffrent ML

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
t = pd.read_csv('outputs/oof_predictions.csv')
te = pd.read_csv('outputs/test_predictions.csv')

blendingmodel = (
    ('lr', LogisticRegression()),
    ('lr', SVC(probability=True))
)
t = pd.read_csv('outputs/oof_predictions.csv')
te = pd.read_csv('outputs/test_predictions.csv')
f_lebels, target_lebel = ['lightgbm', 'catboost', 'xgboost'], 'addicted_label'
for name, bm in blendingmodel:
    bm.fit(t[f_lebels], t[target_lebel])
    pred = bm.predict_proba(te[f_lebels])[:, 1]
    submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], TARGET: pred})
    assert submission.shape == sample_submission.shape
    assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
    assert submission[TARGET].between(0, 1).all()

    submission_path = f'{OUTPUT_DIR}/{name}_submission_blend.csv'
    submission.to_csv(submission_path, index=False)


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [28]:
submission

,id,addicted_label
0,691369,0.975571
1,691370,0.961202
2,691371,0.961638
3,691372,0.974228
4,691373,0.975332
...,...,...
296297,987666,0.975668
296298,987667,0.956413
296299,987668,0.120079
296300,987669,0.895453


In [29]:
te[f_lebels]

,lightgbm,catboost,xgboost
0,0.999363,0.999650,0.999466
1,0.962222,0.943632,0.905366
2,0.927695,0.937631,0.940096
3,0.990793,0.985120,0.993438
4,0.997719,0.996856,0.998469
...,...,...,...
296297,0.999999,0.999989,0.999999
296298,0.923181,0.891721,0.912260
296299,0.193143,0.147536,0.226539
296300,0.732422,0.716731,0.839146


In [22]:
pred[:, 1]

array([0.97557084, 0.96120174, 0.96163803, ..., 0.12007899, 0.89545337,
       0.9171743 ], shape=(296302,))

In [25]:
pred

array([[0.02442916, 0.97557084],
       [0.03879826, 0.96120174],
       [0.03836197, 0.96163803],
       ...,
       [0.87992101, 0.12007899],
       [0.10454663, 0.89545337],
       [0.0828257 , 0.9171743 ]], shape=(296302, 2))